# Analyse classification

## 1 - Environnement

### 1.1 - Dépendances

In [ ]:
# pip

%pip install --upgrade pip

In [ ]:
# installation des dépendances

%pip install --upgrade sys
%pip install --upgrade os

%pip install --upgrade pathlib

%pip install --upgrade importlib

%pip install --upgrade numpy
%pip install --upgrade pandas

%pip install --upgrade shapely

%pip install --upgrade ast

%pip install --upgrade logging

In [ ]:
# chargement des dépendances

import sys
import os

from pathlib import Path

import importlib

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import wkt
from shapely.geometry import Point
from shapely.geometry import Polygon

import ast

import logging

In [ ]:
# configuration du logging

logging.basicConfig(
    level=logging.INFO,  
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

In [ ]:
# chemin relatif depuis ce notebook jusqu'au dossier parent de modules 

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

### 1.2 - Classes 

In [ ]:
import modules.ClassificationUtils as ClassificationUtils
import modules.Utils as Utils

importlib.reload(ClassificationUtils)
importlib.reload(Utils)

## 2 - Contexte

## 3 - Analyse

### 3.1 - Découpage existant

In [ ]:
# chemin vers classification

classification_arrondissement = "../../data/classification/AmsterdamCity/Classification_AmsterdamCity-District.csv"

In [ ]:
# chemin vers demande

chemin_demande_matsim = "../../inputs/Matsim_7-9/Matsim_7-9_Calibration__Original/Matsim_7-9_Calibration__Original__None_None__None_None__None__0.00__0_0.00__0.00.csv"
chemin_demande_amsterdam = "../../inputs/AmsterdamCity_7-9/AmsterdamCity_7-9_Original__Original/AmsterdamCity_7-9_Original__Original__None_None__None_None__None__0.00__0_0.00__0.00.csv"

In [ ]:
# classification

cluster_id_matsim = ClassificationUtils.associate(chemin_demande_matsim, classification_arrondissement, "epsg:32631", False)
cluster_id_amsterdam = ClassificationUtils.associate(chemin_demande_amsterdam, classification_arrondissement, "epsg:32631", False)

In [ ]:
# dataframe et colonne CLUSTER_ID

demande_matsim = pd.read_csv(chemin_demande_matsim, sep=';')
demande_amsterdam = pd.read_csv(chemin_demande_amsterdam, sep=';')

demande_matsim["CLUSTER_ID"] = cluster_id_matsim
demande_amsterdam["CLUSTER_ID"] = cluster_id_amsterdam

In [ ]:
# Cluster ordonné

matsim_classification_arrondissement_ord = ClassificationUtils.ordered_classification(demande_matsim)
amsterdam_classification_arrondissement_ord = ClassificationUtils.ordered_classification(demande_amsterdam)

In [ ]:
# affichage classification ordonnée matsim

from pyproj import Transformer

def get_extent_utm(lon_min, lat_min, lon_max, lat_max, tgt_epsg="EPSG:32631"):
    """
    Convertit une bounding‑box donnée en (lon, lat) WGS84 en coordonnées projetées (x, y) pour un CRS cible.
    Retourne (xmin, xmax, ymin, ymax).
    """
    transformer = Transformer.from_crs("EPSG:4326", tgt_epsg, always_xy=True)

    # Quatre coins
    x1, y1 = transformer.transform(lon_min, lat_min)
    x2, y2 = transformer.transform(lon_min, lat_max)
    x3, y3 = transformer.transform(lon_max, lat_min)
    x4, y4 = transformer.transform(lon_max, lat_max)

    xmin = min(x1, x2, x3, x4)
    xmax = max(x1, x2, x3, x4)
    ymin = min(y1, y2, y3, y4)
    ymax = max(y1, y2, y3, y4)

    return xmin, xmax, ymin, ymax



lon_min, lat_min = 4.70, 52.25
lon_max, lat_max = 5.05, 52.45

extent = get_extent_utm(lon_min, lat_min, lon_max, lat_max)



ClassificationUtils.plot_ordered_classification(matsim_classification_arrondissement_ord.head(10), classification_arrondissement, extent=extent)

In [ ]:
from pyproj import Transformer

def get_extent_utm(lon_min, lat_min, lon_max, lat_max, tgt_epsg="EPSG:32631"):
    """
    Convertit une bounding‑box donnée en (lon, lat) WGS84 en coordonnées projetées (x, y) pour un CRS cible.
    Retourne (xmin, xmax, ymin, ymax).
    """
    transformer = Transformer.from_crs("EPSG:4326", tgt_epsg, always_xy=True)

    # Quatre coins
    x1, y1 = transformer.transform(lon_min, lat_min)
    x2, y2 = transformer.transform(lon_min, lat_max)
    x3, y3 = transformer.transform(lon_max, lat_min)
    x4, y4 = transformer.transform(lon_max, lat_max)

    xmin = min(x1, x2, x3, x4)
    xmax = max(x1, x2, x3, x4)
    ymin = min(y1, y2, y3, y4)
    ymax = max(y1, y2, y3, y4)

    return xmin, xmax, ymin, ymax



lon_min, lat_min = 4.70, 52.25
lon_max, lat_max = 5.05, 52.45

extent = get_extent_utm(lon_min, lat_min, lon_max, lat_max)



ClassificationUtils.plot_ordered_classification(amsterdam_classification_arrondissement_ord.head(48), classification_arrondissement, extent=extent)

In [ ]:
matsim_classification_arrondissement_ord

In [ ]:
df_poly = pd.read_csv(path_polygons_csv, sep=';')